# Logging my journey learning LLMs and PyTorch

It is long overdue for me to start learning PyTorch, and there is no better way to do it than building a toy LLM. I've gone through a list of papers that helped me understand how LLMs come to be, and I think it's a good idea to build a toy LLM to better connect the theory with practice.

# Starting the local colab docker image

Use the following command:
```
docker run -it --gpus=all -p 127.0.0.1:9000:8080 -p 127.0.0.1:6006:6006 --mount type=bind,src="${HOME}/colab/checkpoints",dst=/content/checkpoints us-docker.pkg.dev/colab-images/public/runtime
```

In [ ]:
!pwd && ls

# Setup PyTorch

In [ ]:
import torch

In [ ]:
print(f"PyTorch Version: {torch.__version__}")
if (torch.cuda.is_available()):
    device = torch.device("cuda")
    print(f"Using GPU, CUDA version: {torch.version.cuda}, Number of GPUs: {torch.cuda.device_count()}")
else:
    device = torch.device("cpu")

# Get a working tokenizer
Let's use OpenAI's tiktoken tokenizer - alghough I am actually interested in building a minimum tokenizer from scratch later.

In [ ]:
!git clone https://github.com/openai/tiktoken.git

In [ ]:
import tiktoken

In [ ]:
enc = tiktoken.get_encoding("r50k_base")

In [ ]:
enc.encode("monkey d luffy")

In [ ]:
enc.decode(enc.encode("monkey d luffy"))

# Get the WebText dataset

In [ ]:
from datasets import load_dataset

In [ ]:
openwebtext = load_dataset("openwebtext")

In [ ]:
openwebtext

In [ ]:
openwebtext['train']

## Test tokenizing the training data

In [ ]:
enc.encode(openwebtext['train'][42]['text'])

# Embedding layer

In [ ]:
vocabulary_size = enc.n_vocab
d_model = 768

embedding_layer = torch.nn.Embedding(vocabulary_size, d_model)

Here we encode the phrase "monkey d luffy", which is then fed into the embedding_layer,
which produces d_model=768 embeddings, one 768 dimentional vector per token

In [ ]:
embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")]))

In [ ]:
embedding_layer.weight.shape

Here we can see that the embedding layer is rather simple - it is a n_vocab x d_model matrix. So the embedding process is simply taking w[encoding].

## Positional Encoding

In [ ]:
class LearnablePositionalEncoding(torch.nn.Module):
    def __init__(self, max_seq_len, d_model):
        super().__init__()
        self.pos_embeddings = torch.nn.Embedding(max_seq_len, d_model)
    def forward(self, x):
        # x is of shape (batch_size, seq_len, d_model)
        seq_len = x.size(1)

        position_ids = torch.arange(seq_len, dtype=torch.long, device=x.device)

        pos_enc = self.pos_embeddings(position_ids)
        return x + pos_enc


In [ ]:
pos_embed_layer = LearnablePositionalEncoding(1024, d_model)

In [ ]:
pos_embed_layer(embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")])))

# Decoder Only Transformer Layer

This is the key part of the transformer architecture. Each attention layer is made of multi-head attention, normalization, and position-wise feedforward.

In GPT-2, normalization is moved at the beginning of each layer.

In [ ]:
embedding = embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")]))

## Layer Normalization

https://arxiv.org/pdf/1607.06450

In [ ]:
layer_norm = torch.nn.LayerNorm(d_model)

In [ ]:
x = layer_norm(embedding)

In [ ]:
x

In [ ]:
x.shape

## Multi-Head Attention

In [ ]:
n_head = 12

# project into Q, K, V
projection = torch.nn.Linear(d_model, d_model * 3)

In [ ]:
projection(x).shape

In [ ]:
proj = projection(x).view(1, -1, 12, d_model // n_head * 3).permute(0, 2, 1, 3)

In [ ]:
q, k, v = proj.chunk(3, dim=3)

In [ ]:
import math
atten = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_model)

In [ ]:
mask = torch.tril(torch.ones_like(atten[0][0]))

In [ ]:
atten = atten.masked_fill(mask == 0, -float('inf'))

In [ ]:
atten

In [ ]:
softmax = torch.nn.Softmax(dim=-1)

atten = softmax(atten)

In [ ]:
atten

In [ ]:
torch.matmul(atten, v).transpose(1, 2).reshape(1, 4, d_model)

### Putting it together

In [ ]:
import torch.nn.functional as F
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # project into K Q V
        self.input_linear = torch.nn.Linear(d_model, d_model * 3)
    def forward(self, x):
        batch_size, sequence_length, _ = x.size()
        proj = self.input_linear(x).view(batch_size, sequence_length, self.n_heads, self.d_head * 3).permute(0, 2, 1, 3)
        q, k, v = proj.chunk(3, dim=-1)

        # Standard Transformer scaling: 1/sqrt(d_head)
        atten = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        mask = torch.tril(torch.ones_like(atten[0][0]))
        atten = atten.masked_fill(mask == 0, -float('inf'))
        atten = F.softmax(atten, dim=-1)

        return torch.matmul(atten, v).transpose(1, 2).reshape(batch_size, sequence_length, self.d_model)

In [ ]:
multi_head_attention = MultiHeadAttention(d_model, n_head)
multi_head_attention(x)

## Position-wise Feed Forward

In [ ]:
d_hidden = 3072

In [ ]:
class DenseFeedForward(torch.nn.Module):
    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.model = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(d_hidden, d_model)
        )
    def forward(self, x):
        return self.model(x)

In [ ]:
y = multi_head_attention(x)

In [ ]:
dff = DenseFeedForward(d_model, d_hidden)

In [ ]:
dff(y)

## The Full Transformer Layer

In [ ]:
class TransformerDecoderOnly(torch.nn.Module):
    def __init__(self, d_model, d_hidden, n_head):
        super().__init__()
        self.layer_norm_0 = torch.nn.LayerNorm(d_model)
        self.multi_head_attention = MultiHeadAttention(d_model, n_head)
        self.layer_norm_1 = torch.nn.LayerNorm(d_model)
        self.dff = DenseFeedForward(d_model, d_hidden)
    def forward(self, x):
        # layer normalization first
        x = self.layer_norm_0(x)

        # multi-head attention and residual
        identity = x
        x = self.multi_head_attention(x)
        x = x + identity

        # layer normalization before dense feed forward
        x = self.layer_norm_1(x)

        # dense feed forward and residual
        identity = x
        x = self.dff(x)
        return x + identity

In [ ]:
transformer_decoder_only = TransformerDecoderOnly(d_model, d_hidden, n_head)

transformer_decoder_only(embedding)

# Output Layer

The output layer is rather simple. matrix multiply with the input embedding matrix, and soft max. We can then reverse the tokenization.

In [ ]:
torch.matmul(transformer_decoder_only(embedding), embedding_layer.weight.transpose(0, 1))

In [ ]:
class Output(torch.nn.Module):
    def __init__(self, embedding_layer_weights):
        super().__init__()
        # Removed the extra LayerNorm here as it can skew initial logits
        self.embedding_layer_weights = embedding_layer_weights

    def forward(self, x):
        # x: (batch, seq, d_model)
        # weights: (vocab, d_model)
        return torch.matmul(x, self.embedding_layer_weights.transpose(0, 1))

# The full model

In [ ]:
class ToyGPT(torch.nn.Module):
    def __init__(self, d_model, d_hidden, n_head, vocab_size, layers, max_seq_len):
        super().__init__()
        self.embedding_layer = torch.nn.Embedding(vocab_size, d_model)
        self.pos_encoding_layer = LearnablePositionalEncoding(max_seq_len, d_model)
        self.transformers = torch.nn.Sequential(*[TransformerDecoderOnly(d_model, d_hidden, n_head) for _ in range(layers)])
        # Standard GPT-2: Final LayerNorm before the output head
        self.ln_f = torch.nn.LayerNorm(d_model)
        self.output_layer = Output(self.embedding_layer.weight)

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, torch.nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, torch.nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x):
        x = self.embedding_layer(x)
        x = self.pos_encoding_layer(x)
        x = self.transformers(x)
        x = self.ln_f(x) # Apply the missing final normalization
        return self.output_layer(x)

# Training

In [ ]:
batch_size = 48
max_seq_len = 1024

In [ ]:
enc._special_tokens

Pre-tokenize the training data to avoid CPU bottleneck.

In [ ]:
from datasets import Dataset

rows = openwebtext["train"].num_rows
bos_token = '<|endoftext|>'

def tokenize(examples):
    out = enc.encode(examples['text']+ bos_token, allowed_special={bos_token})
    return {'tokenized_text': out}

tokenized_data = openwebtext["train"].map(tokenize, remove_columns=openwebtext["train"].column_names, num_proc=32)


In [ ]:
def do_checkpoint(model, optimizer, running_encoding, row_number):
    torch.save({
        'row_number': row_number,
        'running_encoding': running_encoding,
        'optimizer_state_dict': optimizer.state_dict(),
        'model_state_dict': model.state_dict(),
    }, 'checkpoints/check_point_{}'.format(row_number))

In [ ]:
@torch.no_grad()
def generate_with_sampling(model, prompt="The secret to life is", max_new_tokens=50, temperature=0.8, top_k=40):
    model.eval()
    input_ids = torch.tensor([enc.encode(prompt)], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        cond_ids = input_ids[:, -max_seq_len:]
        logits = model(cond_ids)
        logits = logits[:, -1, :] / temperature

        # Optional: Top-K filtering
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits[logits < v[:, [-1]]] = -float('Inf')

        # Sample from the distribution instead of taking the max
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat((input_ids, next_token), dim=1)
        if next_token.item() == enc._special_tokens[bos_token]:
            break

    out_text = enc.decode(input_ids[0].tolist())
    model.train()
    return out_text

# Try it with your current model scale
# print(generate_with_sampling(model, temperature=0.8, top_k=50))

In [ ]:
import gc
import math
from tqdm.notebook import tqdm

def get_lr(step, warmup_steps, max_steps, max_lr, min_lr):
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    if step > max_steps:
        return min_lr
    decay_ratio = (step - warmup_steps) / (max_steps - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)

@torch.no_grad()
def generate_sample(model, prompt="The secret to life is", max_new_tokens=50):
    model.eval()
    input_ids = torch.tensor([enc.encode(prompt)], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        cond_ids = input_ids[:, -max_seq_len:]
        logits = model(cond_ids)
        logits = logits[:, -1, :]
        next_token = torch.argmax(logits, dim=-1, keepdim=True)
        input_ids = torch.cat((input_ids, next_token), dim=1)
        if next_token.item() == enc._special_tokens[bos_token]:
            break
    out_text = enc.decode(input_ids[0].tolist())
    model.train()
    return out_text

def train(start_row_number = 0, check_point_interval = 100000, summary_writer = None):
    tokens_per_batch = (batch_size + 1) * max_seq_len // 2
    max_lr = 0.0006
    min_lr = max_lr * 0.1
    warmup_steps = 20000
    max_steps = rows

    encoding = []
    model = ToyGPT(d_model, d_hidden, n_head, enc.n_vocab, 12, max_seq_len)
    optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.1)
    model.to(device)
    model.train()
    model = torch.compile(model)

    if start_row_number != 0:
        checkpoint = torch.load('checkpoints/check_point_{}'.format(start_row_number))
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        encoding = checkpoint['running_encoding']

    write_summary_threshold = start_row_number
    check_point_threshold = (start_row_number + check_point_interval) // check_point_interval * check_point_interval
    data = tokenized_data.select(range(start_row_number, tokenized_data.num_rows)).to_iterable_dataset(num_shards=8)

    for row in enumerate(tqdm(data, total=rows, initial=start_row_number)):
        cur_row_number = start_row_number + row[0]
        lr = get_lr(cur_row_number, warmup_steps, max_steps, max_lr, min_lr)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        encoding.extend([enc._special_tokens[bos_token]] + row[1]['tokenized_text'])

        if len(encoding) > tokens_per_batch:
            residual = encoding[tokens_per_batch:]
            encoding = encoding[0:tokens_per_batch + 1]
            batch = torch.empty((batch_size, max_seq_len), dtype=torch.long, device=device)
            target = torch.empty((batch_size, max_seq_len), dtype=torch.long, device=device)

            step_size = max_seq_len // 2
            for i in range(0, batch_size):
                start_idx = i * step_size
                batch[i] = torch.LongTensor(encoding[start_idx : start_idx + max_seq_len])
                target[i] = torch.LongTensor(encoding[start_idx + 1 : start_idx + max_seq_len + 1])

            optimizer.zero_grad()
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                out = model(batch)
                loss = F.cross_entropy(out.view(-1, enc.n_vocab), target.view(-1))

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            encoding = residual

            if summary_writer is not None and cur_row_number >= write_summary_threshold:
                write_summary_threshold = (write_summary_threshold + 1000) // 1000 * 1000
                summary_writer.add_scalar('Loss/train', loss.item(), cur_row_number)
                summary_writer.add_scalar('LR/train', lr, cur_row_number)
                summary_writer.flush()

            if (cur_row_number + 1 >= check_point_threshold):
                print(f"\nCheckpointing at {cur_row_number + 1}. Loss: {loss.item():.4f}")
                do_checkpoint(model, optimizer, encoding, cur_row_number + 1)

                # Generate and log sample
                sample = generate_sample(model)
                print(f"Sample Generation: {sample}\n")
                sample_with_temperature = generate_with_sampling(model, temperature=0.8, top_k=50)
                print(f"Sample Generation with temperature 0.8: {sample_with_temperature}")
                if summary_writer is not None:
                    summary_writer.add_text('Generation/sample', sample, cur_row_number + 1)
                    summary_writer.add_text('Generation/sample_with_temp', sample_with_temperature, cur_row_number + 1)
                    summary_writer.flush()

                check_point_threshold = (cur_row_number + 1 + check_point_interval) // check_point_interval * check_point_interval

            gc.collect()

### Validation: Architecture and Initial Loss
For a randomly initialized model, the expected cross-entropy loss is $-\ln(1/V) = \ln(V)$, where $V$ is the vocabulary size. Let's verify this and check the output shapes.

In [ ]:
import math

# 1. Check Output Shapes
test_model = ToyGPT(d_model, d_hidden, n_head, vocabulary_size, layers=1, max_seq_len=max_seq_len)
test_input = torch.randint(0, vocabulary_size, (1, 8)) # Batch size 1, Seq len 8
with torch.no_grad():
    logits = test_model(test_input)

print(f"Input shape: {test_input.shape}")
print(f"Output (logits) shape: {logits.shape} (Expected: [1, 8, {vocabulary_size}])")

# 2. Check Initial Loss
targets = torch.randint(0, vocabulary_size, (1, 8))
loss = F.cross_entropy(logits.view(-1, vocabulary_size), targets.view(-1))

theoretical_loss = math.log(vocabulary_size)
print(f"\nTheoretical Initial Loss: {theoretical_loss:.4f}")
print(f"Actual Initial Loss:      {loss.item():.4f}")
print(f"Difference:              {abs(loss.item() - theoretical_loss):.4f}")

if abs(loss.item() - theoretical_loss) < 0.5:
    print("\n✅ The initial loss is consistent with random initialization.")
else:
    print("\n⚠️ Initial loss is significantly different from theory. Check initialization or scaling.")

In [ ]:
%load_ext tensorboard

In [ ]:
from torch.utils.tensorboard import SummaryWriter

In [ ]:
%tensorboard --logdir checkpoints/adamw_run_lr_0006 --bind_all


In [ ]:
summary_writer = SummaryWriter('./checkpoints/adamw_run_lr_0006')

train(start_row_number=7400004, check_point_interval=100000, summary_writer = summary_writer)